# P1 – Fusion-LR (Sơn)

Same logistic regression as the baselines, with content and graph features joined: P + T + G-struct.
G-emb (Node2Vec) is added later, when PyTorch Geometric is installed.
The row "P1 − G-struct" (= P + T) shows what the graph features add (ablation A2).

In [1]:
# 1. Setup (Colab or local)
import os, sys
if not os.path.exists("mgtab.py"):
    if "google.colab" in sys.modules:
        if not os.path.exists("sna-content-moderation"):
            !git clone -q https://github.com/ThSonDev/sna-content-moderation.git
        %cd sna-content-moderation
        !pip install -q -r requirements.txt
    else:
        %cd ..
sys.path.insert(0, os.getcwd())

/mnt/DevZone/social-networks/project/sna-content-moderation


In [2]:
# 2. Download MGTAB (31 MB) into data/MGTAB, only once
import urllib.request, zipfile
if not os.path.exists("data/MGTAB/features.pt"):
    os.makedirs("data", exist_ok=True)
    url = "https://drive.usercontent.google.com/download?id=1gbWNOoU1JB8RrTu2a5j9KMNVa9wX72Fe&export=download&confirm=t"
    urllib.request.urlretrieve(url, "data/MGTAB.zip")
    zipfile.ZipFile("data/MGTAB.zip").extractall("data")

In [3]:
# 3. Load data and the shared split (never make your own split)
import numpy as np
from mgtab import load, relations, split, best_threshold, evaluate, RELATIONS
X, y, stance, E, T, W = load()
rel = relations(E, T, len(y))   # {"follow": (src, dst), ...}
train, val, test = split(y)     # test stays locked until the final run
print(X.shape, {r: len(s) for r, (s, d) in rel.items()})

(10199, 788) {'follow': 425863, 'mention': 110868, 'reply': 22327, 'quote': 20901, 'url': 263800, 'hashtag': 300000}


## Features

In [4]:
from sklearn.preprocessing import StandardScaler
from features import struct_features, PROFILE_ONLY
G, g_names = struct_features(X, rel)            # 32 cols, cached in data/ after the first run
G = StandardScaler().fit(G[train]).transform(G)  # train statistics only
P, Tw = X[:, PROFILE_ONLY], X[:, 20:]           # profile (16, scaled by the authors), tweets (768)
groups = {"P1 (P + T + G-struct)": np.hstack([P, Tw, G]), "P1 − G-struct (P + T)": np.hstack([P, Tw])}

## Train and evaluate (validation only)

C is tuned on validation PR-AUC; the threshold is tuned on validation macro-F1.

In [5]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

def fusion_lr(Z):
    best = None
    for C in [0.01, 0.1, 1, 10]:
        clf = LogisticRegression(C=C, class_weight="balanced", max_iter=5000).fit(Z[train], y[train])
        s = clf.predict_proba(Z)[:, 1]
        ap = average_precision_score(y[val], s[val])
        if best is None or ap > best[0]:
            best = (ap, C, s)
    _, C, s = best
    return {"C": C, **evaluate(y[val], s[val], best_threshold(y[val], s[val]))}

res = pd.DataFrame({name: fusion_lr(Z) for name, Z in groups.items()}).T
res.to_csv("results/p1.csv")
res.round(3)

,C,pr_auc,roc_auc,macro_f1,p@100,p@200
P1 (P + T + G-struct),0.1,0.914,0.971,0.908,0.97,0.935
P1 − G-struct (P + T),0.1,0.821,0.930,0.840,0.91,0.890
